# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load, inspect, and process a FAIR<sup>2</sup> dataset using the `mlcroissant` library. We will explore the dataset schema, extract records using entity `@id`s, and perform exploratory analysis.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata using `mlcroissant`. The schema is loaded from the Croissant JSON-LD URL.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Print basic metadata (as attributes)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Browse record sets and their field (column) `@id`s. All entities are referenced by their `@id` per the Croissant schema.

In [ ]:
# List all available record sets and their fields, by @id

record_sets = dataset.record_sets
print(f"Number of record sets: {len(record_sets)}")

for rs in record_sets:
    print(f"\nRecord Set '@id': {rs.id}")
    print(f"  Name: {rs.name}")
    print("  Fields/Columns:")
    for field in rs.fields:
        print(f"    - @id: {field.id} (name: {field.name}, dataType: {getattr(field, 'data_type', None)})")

## 3. Data Extraction
Load records from a specific record set into a pandas DataFrame for analysis. All record set and column references use the entity `@id`s discovered above.

In [ ]:
# Select primary record set(s) (by @id) for extraction; here, using first one as example
rs_ids = [rs.id for rs in dataset.record_sets]
print(f"Record set @id(s) detected: {rs_ids}")

dataframes = {}
for rs_id in rs_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for Record Set '@id': {rs_id} (shape={df.shape})")

# Display a sample of columns in the first record set DataFrame loaded
main_rs_id = rs_ids[0]
print(f"\nFirst record set '@id': {main_rs_id}")
print("Columns:", dataframes[main_rs_id].columns.tolist())
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Let's process and explore numeric fields by their `@id`. We'll filter on a chosen numeric column, normalize, and group by a categorical column, all referenced by `@id`.

In [ ]:
# Choose a numeric and a categorical field by their @id from earlier overview step.
# You may need to update these IDs depending on the dataset schema. Here we print and select candidates:

df = dataframes[main_rs_id]
print("Column @ids and sample values:")
for c in df.columns:
    print(f"- {c}: type={df[c].dtype}, sample={df[c].head(2).tolist()}")

# Replace below with actual @ids for a numeric field and a grouping field (categorical)
# For demonstration, we'll search for 'age', 'interval', or similar in the column names

numeric_field_id = next((c for c in df.columns if 'age' in c.lower() or 'interval' in c.lower() or df[c].dtype.kind in 'iufc'), None)
group_field_id = next((c for c in df.columns if 'sex' in c.lower() or 'gender' in c.lower() or 'site' in c.lower() or 'anatomical' in c.lower()), None)

print(f"\nSelected numeric field @id: {numeric_field_id}")
print(f"Selected group (categorical) field @id: {group_field_id}")

if numeric_field_id is not None:
    # Try to convert values to numeric if not already
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].mean()  # Use mean as a sample filter
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (n={len(filtered_df)}):")
    display_cols = [numeric_field_id]
    print(filtered_df[display_cols].head())

    # Normalize
    normalized_col = f"{numeric_field_id}_normalized"
    filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records (first 5):")
    print(filtered_df[[numeric_field_id, normalized_col]].head())

    # Group by group_field_id (if exists)
    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean of {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df)
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Let's visualize the (filtered) numeric column and group distributions.<br>All axes and legends reference column `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion

- We loaded and explored the FAIR<sup>2</sup> colorectal clinicopathological dataset using entity `@id` for all accesses.
- We reviewed available record sets, fields, and loaded main tabular records into a DataFrame.
- Numeric and categorical columns were selected (referenced by `@id`), filtered, normalized, grouped and visualized.
- Further analyses could include advanced statistical methods or ML modeling, referencing dataset schema by `@id` throughout.